> ## ⚠️  Superseded — single-run draft, retained as a record
>
> **This notebook will not run.** It opens `processed_data/all_data.pkl`, the earlier
> 53-feature preprocessing artefact that was replaced by `canonical_data.pkl` when the
> feature set was corrected. Nothing in this folder regenerates it, so the first cell raises
> `FileNotFoundError`.
>
> **No result reported in the thesis comes from this notebook.** Every reported figure is
> produced by the canonical path `00`/`01` → `08`–`15`, which trains over thirty fixed seeds
> (`SEEDS = [42 + 7*i for i in range(30)]`). Model definitions — `TemporalAttention`,
> `build_lstm_transformer`, `build_standalone_lstm` — live in `canonical_pipeline.py`.
>
> It is kept because its saved outputs are the record of the single-run baseline stage the
> project passed through. See README §5.

# 06b — RF-Enhanced LSTM-Transformer (Proposed Modification)

## Architecture Modification (Section 3.2)

Following the methodology in Section 3.2.1 (Proposed Solution — Process and Flow), we **add a Random Forest node** into the LSTM-Transformer pipeline:

**Original LSTM-Transformer:**
```
Input Sequence → LSTM → Attention → Pool ─┐
                                           ├─ Concat → Dense → Output
Current Features → Dense ─────────────────┘
```

**Modified RF-Enhanced LSTM-Transformer:**
```
Input Sequence → LSTM → Attention → Pool ─┐
                                           ├─ Concat(3-way) → Dense → Output  
Current Features → Dense ─────────────────┤
RF Prediction → Dense ───────────────────┘  ← NEW ADDED NODE
```

The RF prediction is treated as an **additional input feature** that carries the strong autocorrelation signal RF is good at extracting, while LSTM-Transformer adds temporal pattern recognition and interpretability via attention weights.

This follows the green flowchart pattern from the methodology: adding a node from Model B (Random Forest) into Model A (LSTM-Transformer).

In [1]:
import pickle, numpy as np, warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
np.random.seed(42); tf.random.set_seed(42)

with open('processed_data/all_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train_scaled = data['X_train_scaled']
X_test_scaled = data['X_test_scaled']
y_train = data['y_train']
y_test = data['y_test']
X_tr_seq_ = data['X_tr_seq_']
X_tr_feat_ = data['X_tr_feat_']
y_tr_seq_ = data['y_tr_seq_']
X_val_seq = data['X_val_seq']
X_val_feat = data['X_val_feat']
y_val_seq = data['y_val_seq']
X_te_seq = data['X_te_seq']
X_te_feat = data['X_te_feat']
y_te_seq_raw = data['y_te_seq_raw']
target_scaler = data['target_scaler']
SEQ_LEN = data['SEQ_LEN']
n_features = X_tr_seq_.shape[2]

print(f"Data loaded: train={len(X_tr_seq_)}, val={len(X_val_seq)}, test={len(X_te_seq)}")

Data loaded: train=2431, val=429, test=695


## Step 1: Train Random Forest (the "added node")

RF is trained first. Its predictions become an additional input to the LSTM-Transformer.

In [2]:
# Train RF on flat features
print("Step 1: Training Random Forest...")
rf_model = RandomForestRegressor(
    n_estimators=300, max_depth=12,
    min_samples_split=10, min_samples_leaf=5,
    max_features='sqrt', random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

# Generate RF predictions for ALL data (train + test)
rf_pred_train = rf_model.predict(X_train_scaled)
rf_pred_test = rf_model.predict(X_test_scaled)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred_test))
rf_r2 = r2_score(y_test, rf_pred_test)
print(f"RF standalone — RMSE: {rf_rmse:.2f}, R²: {rf_r2:.4f}")

# Scale RF predictions to [0,1] same as target
rf_pred_train_scaled = target_scaler.transform(rf_pred_train.reshape(-1,1)).flatten()
rf_pred_test_scaled = target_scaler.transform(rf_pred_test.reshape(-1,1)).flatten()
print("✅ RF predictions generated and scaled")

Step 1: Training Random Forest...
RF standalone — RMSE: 13.94, R²: 0.9375
✅ RF predictions generated and scaled


## Step 2: Create RF-enhanced sequences

Each training/test sample now has a **third input**: the RF prediction for that day.

In [3]:
# Create RF prediction sequences aligned with DL sequences
# The sequence at index i uses features[i:i+SEQ_LEN], predicting target[i+SEQ_LEN]
# So the RF prediction for sample i is rf_pred_scaled[i+SEQ_LEN]

def create_rf_input(rf_pred_scaled, n_samples, seq_len, total_len):
    """Extract RF predictions aligned with sequence targets."""
    rf_input = []
    for i in range(total_len - seq_len):
        rf_input.append(rf_pred_scaled[i + seq_len])
    return np.array(rf_input[:n_samples]).reshape(-1, 1)

# Training RF inputs
n_train_total = len(X_tr_seq_) + len(X_val_seq)
rf_all_train = create_rf_input(rf_pred_train_scaled, n_train_total, SEQ_LEN, len(rf_pred_train_scaled))
rf_tr = rf_all_train[:len(X_tr_seq_)]
rf_val = rf_all_train[len(X_tr_seq_):len(X_tr_seq_)+len(X_val_seq)]

# Test RF inputs
rf_te = create_rf_input(rf_pred_test_scaled, len(X_te_seq), SEQ_LEN, len(rf_pred_test_scaled))

print(f"RF inputs: train={rf_tr.shape}, val={rf_val.shape}, test={rf_te.shape}")
print(f"RF prediction range: [{rf_tr.min():.3f}, {rf_tr.max():.3f}]")

RF inputs: train=(2431, 1), val=(429, 1), test=(695, 1)
RF prediction range: [0.002, 0.437]


## Step 3: AttentionExtractor layer

In [4]:
class AttentionExtractor(layers.Layer):
    """Wraps MultiHeadAttention and stores attention weights for extraction."""
    def __init__(self, num_heads, key_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim, dropout=dropout)
        self.attention_weights = None
    def call(self, query, value, training=False):
        output, weights = self.mha(query, value, return_attention_scores=True, training=training)
        self.attention_weights = weights
        return output
    def get_config(self):
        config = super().get_config()
        config.update({'num_heads': self.mha.num_heads, 'key_dim': self.mha._key_dim, 'dropout': self.mha._dropout})
        return config
print("✅ AttentionExtractor defined")

✅ AttentionExtractor defined


## Step 4: Build RF-Enhanced LSTM-Transformer

The key modification: **3 input branches** instead of 2.

```
Branch 1: Sequence → LSTM(16) → Attention(2h,8d) → Pool → 16-dim
Branch 2: Current features → Dense(16) → 16-dim  
Branch 3: RF prediction → Dense(8) → 8-dim         ← NEW NODE from RF
                    ↓
            Concat(16+16+8=40) → Dense(16) → Output
```

In [5]:
# ═══ RF-Enhanced LSTM-Transformer ═══
# 3 inputs: sequence, current features, RF prediction

si = Input(shape=(SEQ_LEN, n_features), name='sequence_input')
fi = Input(shape=(n_features,), name='feature_input')
ri = Input(shape=(1,), name='rf_prediction_input')  # ← NEW: RF prediction

# Branch 1: LSTM → Attention (same as original)
x = layers.LSTM(16, return_sequences=True, dropout=0.2, name='lstm_encoder')(si)
x = layers.LayerNormalization(name='lstm_ln')(x)
attn_layer = AttentionExtractor(num_heads=2, key_dim=8, dropout=0.1, name='attention_extractor')
attn_out = attn_layer(x, x)
attn_out = layers.LayerNormalization(name='attn_ln')(attn_out + x)
attn_pooled = layers.GlobalAveragePooling1D(name='attn_pool')(attn_out)

# Branch 2: Feature branch (same as original)
feat = layers.Dense(16, activation='relu', name='feat_dense')(fi)
feat = layers.Dropout(0.2, name='feat_do')(feat)

# Branch 3: RF prediction branch (NEW ADDED NODE)
rf_branch = layers.Dense(8, activation='relu', name='rf_dense')(ri)
rf_branch = layers.Dropout(0.1, name='rf_do')(rf_branch)

# 3-way concat: attention + features + RF
combined = layers.Concatenate(name='merge_3way')([attn_pooled, feat, rf_branch])

# Prediction head
h = layers.Dense(16, activation='relu', name='head_dense')(combined)
h = layers.Dropout(0.2, name='head_do')(h)
o = layers.Dense(1, activation='sigmoid', name='forecast')(h)

model = Model(inputs=[si, fi, ri], outputs=o, name='RF_Enhanced_LSTM_Transformer')
model.attn_layer = attn_layer
model.compile(optimizer=Adam(learning_rate=0.001, clipnorm=1.0), loss='mse', metrics=['mae'])

print(f"RF-Enhanced LSTM-Transformer: {model.count_params():,} params")
print(f"  vs original LSTM-Transformer: 7,041 params")
print(f"  Added params from RF branch: ~{model.count_params()-7041}")
model.summary()


RF-Enhanced LSTM-Transformer: 7,185 params
  vs original LSTM-Transformer: 7,041 params
  Added params from RF branch: ~144


Model: "RF_Enhanced_LSTM_Transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 28, 53)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_encoder (LSTM) │ (None, 28, 16)    │      4,480 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_ln             │ (None, 28, 16)    │         32 │ lstm_encoder[0][… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_extractor │ (None, 28, 16)    │      1,088 │ lstm_ln[0][0],    │
│ (AttentionExtracto… │                   │            │ lstm_ln[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 28, 16)    │          0 │ attention_extrac… │
│                     │                   │            │ lstm_ln[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_input       │ (None, 53)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rf_prediction_input │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_ln             │ (None, 28, 16)    │         32 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feat_dense (Dense)  │ (None, 16)        │        864 │ feature_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rf_dense (Dense)    │ (None, 8)         │         16 │ rf_prediction_in… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_pool           │ (None, 16)        │          0 │ attn_ln[0][0]     │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feat_do (Dropout)   │ (None, 16)        │          0 │ feat_dense[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rf_do (Dropout)     │ (None, 8)         │          0 │ rf_dense[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ merge_3way          │ (None, 40)        │          0 │ attn_pool[0][0],  │
│ (Concatenate)       │                   │            │ feat_do[0][0],    │
│                     │                   │            │ rf_do[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ head_dense (Dense)  │ (None, 16)        │        656 │ merge_3way[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ head_do (Dropout)   │ (None, 16)        │          0 │ head_dense[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ forecast (Dense)    │ (None, 1)         │         17 │ head_do[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,185 (28.07 KB)

 Trainable params: 7,185 (28.07 KB)

 Non-trainable params: 0 (0.00 B)

## Step 5: Train

In [6]:
print("Training RF-Enhanced LSTM-Transformer...")
_ = model.predict([X_tr_seq_[:1], X_tr_feat_[:1], rf_tr[:1]], verbose=0)  # warmup

history = model.fit(
    [X_tr_seq_, X_tr_feat_, rf_tr], y_tr_seq_,
    validation_data=([X_val_seq, X_val_feat, rf_val], y_val_seq),
    epochs=100, batch_size=32, verbose=0,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-7, verbose=0),
    ])

pred_s = model.predict([X_te_seq, X_te_feat, rf_te], verbose=0).flatten()
pred = np.maximum(target_scaler.inverse_transform(pred_s.reshape(-1,1)).flatten(), 0)
rmse = np.sqrt(mean_squared_error(y_te_seq_raw, pred))
mae = mean_absolute_error(y_te_seq_raw, pred)
r2 = r2_score(y_te_seq_raw, pred)

best = np.argmin(history.history['val_loss']) + 1
print(f"\n✅ RF-Enhanced LSTM-Transformer")
print(f"   Best epoch: {best}/{len(history.history['val_loss'])}")
print(f"   RMSE: {rmse:.2f}, MAE: {mae:.2f}, R²: {r2:.4f}")
print(f"\n   vs RF standalone:        RMSE={rf_rmse:.2f}, R²={rf_r2:.4f}")
print(f"   vs original LSTM-T:      RMSE=27.61, R²=0.7616")
print(f"   Improvement over RF:     {(rf_rmse-rmse)/rf_rmse*100:+.1f}%")
print(f"   Improvement over LSTM-T: {(27.61-rmse)/27.61*100:+.1f}%")

Training RF-Enhanced LSTM-Transformer...
Epoch 39: early stopping
Restoring model weights from the end of the best epoch: 19.

✅ RF-Enhanced LSTM-Transformer
   Best epoch: 19/39
   RMSE: 26.87, MAE: 15.20, R²: 0.7742

   vs RF standalone:        RMSE=13.94, R²=0.9375
   vs original LSTM-T:      RMSE=27.61, R²=0.7616
   Improvement over RF:     -92.7%
   Improvement over LSTM-T: +2.7%


## Step 6: Extract attention weights

In [7]:
def extract_attention_weights(model, X_seq, X_feat, X_rf, batch_size=64):
    all_weights = []
    for i in range(0, len(X_seq), batch_size):
        batch_seq = tf.constant(X_seq[i:i+batch_size], dtype=tf.float32)
        batch_feat = tf.constant(X_feat[i:i+batch_size], dtype=tf.float32)
        batch_rf = tf.constant(X_rf[i:i+batch_size], dtype=tf.float32)
        _ = model([batch_seq, batch_feat, batch_rf], training=False)
        all_weights.append(model.attn_layer.attention_weights.numpy())
    return np.concatenate(all_weights, axis=0)

attn_weights = extract_attention_weights(model, X_te_seq, X_te_feat, rf_te)
print(f"Attention weights shape: {attn_weights.shape}")

temporal_importance = attn_weights.mean(axis=(0, 1, 2))
feature_importance = np.zeros(n_features)
for t in range(SEQ_LEN):
    feature_importance += temporal_importance[t] * np.abs(X_te_seq[:, t, :]).mean(axis=0)
feature_importance /= feature_importance.max()
print(f"✅ Attention weights extracted")

Attention weights shape: (695, 2, 28, 28)
✅ Attention weights extracted


## Step 7: Save results

In [8]:
results = {
    'model': 'RF-Enhanced LSTM-Transformer',
    'RMSE': rmse, 'MAE': mae, 'R2': r2,
    'predictions': pred, 'history': history.history,
    'rf_standalone_rmse': rf_rmse, 'rf_standalone_r2': rf_r2,
}
with open('metrics/rf_enhanced_lstm_transformer_results.pkl', 'wb') as f:
    pickle.dump(results, f)
model.save('models/rf_enhanced_lstm_transformer.keras')
np.save('attention/rf_enhanced_weights.npy', attn_weights)
np.save('attention/rf_enhanced_temporal_importance.npy', temporal_importance)
np.save('attention/rf_enhanced_feature_importance.npy', feature_importance)

# Also save the RF model used
with open('models/rf_for_hybrid.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

print("✅ All saved:")
print("  models/rf_enhanced_lstm_transformer.keras")
print("  models/rf_for_hybrid.pkl")
print("  metrics/rf_enhanced_lstm_transformer_results.pkl")
print("  attention/rf_enhanced_*.npy")

✅ All saved:
  models/rf_enhanced_lstm_transformer.keras
  models/rf_for_hybrid.pkl
  metrics/rf_enhanced_lstm_transformer_results.pkl
  attention/rf_enhanced_*.npy


## Summary

| Architecture | RMSE | R² | Key Change |
|---|---|---|---|
| SARIMA | 62.52 | -0.257 | — |
| Vanilla LSTM | 31.12 | 0.697 | — |
| Vanilla Transformer | 29.45 | 0.729 | — |
| LSTM-Transformer (original) | 27.61 | 0.762 | — |
| Random Forest (standalone) | 13.94 | 0.938 | — |
| **RF-Enhanced LSTM-T** | **??.??** | **?.???** | **Added RF node** |

The RF-Enhanced model should outperform the original LSTM-Transformer because it now has access to RF's strong autocorrelation signal as an additional input, while still maintaining the attention-based interpretability.